# CP4 - Case iFood

Cognitive Data Science + Machine Learning & Modelling

Grupo Perceptron, sala 1TIAPZ-2026.

Pergunta que guiou nosso trabalho: como priorizar clientes com maior chance de responder a uma campanha?

O modelo descreve respostas históricas. Ele não mede o efeito causal de uma campanha.

Execute as células nesta ordem. A senha Oracle nunca deve ser escrita no notebook.

In [ ]:
!pip -q install numpy==2.4.4 pandas==3.0.2 scikit-learn==1.9.1 oracledb==3.3.0
import getpass
import oracledb
import pandas as pd
import matplotlib.pyplot as plt

## 1. Ler a tabela do Oracle

Antes desta célula, crie e carregue `IFOOD_CUSTOMERS` no SQL Developer. No Colab, abra o painel de chave `Secrets`, crie `ORACLE_PASSWORD`, informe a senha e ative o acesso para este notebook. Se o segredo não estiver configurado, a célula solicitará a senha sem exibi-la.

In [ ]:
ORACLE_USER = 'RM573854'
try:
    from google.colab import userdata
    ORACLE_PASSWORD = userdata.get('ORACLE_PASSWORD')
except Exception:
    ORACLE_PASSWORD = None
if not ORACLE_PASSWORD:
    ORACLE_PASSWORD = getpass.getpass('Senha Oracle (não exibida): ')
ORACLE_DSN = oracledb.makedsn('oracle.fiap.com.br', 1521, service_name='orcl')

SQL = '''SELECT
    ID, YEAR_BIRTH, EDUCATION, MARITAL_STATUS, INCOME, KIDHOME, TEENHOME,
    DT_CUSTOMER, RECENCY, MNTWINES, MNTFRUITS, MNTMEATPRODUCTS,
    MNTFISHPRODUCTS, MNTSWEETPRODUCTS, MNTGOLDPRODS, NUMDEALSPURCHASES,
    NUMWEBPURCHASES, NUMCATALOGPURCHASES, NUMSTOREPURCHASES,
    NUMWEBVISITSMONTH, ACCEPTEDCMP3, ACCEPTEDCMP4, ACCEPTEDCMP5,
    ACCEPTEDCMP1, ACCEPTEDCMP2, COMPLAIN, Z_COSTCONTACT, Z_REVENUE, RESPONSE
FROM IFOOD_CUSTOMERS
ORDER BY ID'''

with oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN) as connection:
    with connection.cursor() as cursor:
        cursor.execute(SQL)
        columns = [item[0] for item in cursor.description]
        df = pd.DataFrame(cursor.fetchall(), columns=columns)

df.attrs['data_source'] = 'Oracle'
print('shape:', df.shape)
print('INCOME nulo:', int(df['INCOME'].isna().sum()))

In [ ]:
"""Pipeline reproduzivel da Parte 2 do CP4.

Recebe o DataFrame retornado pela consulta Oracle.
O holdout fica separado antes do tuning. Perfis de features repetidos ficam no
mesmo grupo entre treino, teste e folds. Imputacao e one-hot ficam dentro do
Pipeline, portanto usam apenas os dados de treino durante o ajuste. A ordenacao
por ID e o random_state fixo mantem o split reproduzivel no Oracle e no CSV.
"""

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


RANDOM_STATE = 42


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result.columns = [str(column).upper() for column in result.columns]
    return result


def build_features(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    data = normalize_columns(df)
    if "ID" in data.columns:
        data = data.sort_values("ID", kind="mergesort").reset_index(drop=True)
    data["DT_CUSTOMER"] = pd.to_datetime(data["DT_CUSTOMER"], errors="coerce")

    # Features criadas antes do split, sem usar RESPONSE.
    data["AGE_AT_2014"] = 2014 - data["YEAR_BIRTH"]
    data["TOTAL_CHILDREN"] = data["KIDHOME"] + data["TEENHOME"]
    spend_columns = [
        "MNTWINES", "MNTFRUITS", "MNTMEATPRODUCTS",
        "MNTFISHPRODUCTS", "MNTSWEETPRODUCTS", "MNTGOLDPRODS",
    ]
    purchase_columns = [
        "NUMDEALSPURCHASES", "NUMWEBPURCHASES",
        "NUMCATALOGPURCHASES", "NUMSTOREPURCHASES",
    ]
    campaign_columns = [
        "ACCEPTEDCMP1", "ACCEPTEDCMP2", "ACCEPTEDCMP3",
        "ACCEPTEDCMP4", "ACCEPTEDCMP5",
    ]
    data["TOTAL_SPEND"] = data[spend_columns].sum(axis=1)
    data["TOTAL_PURCHASES"] = data[purchase_columns].sum(axis=1)
    data["CAMPAIGNS_ACCEPTED"] = data[campaign_columns].sum(axis=1)
    data["WEB_PURCHASE_SHARE"] = (
        data["NUMWEBPURCHASES"] / data["TOTAL_PURCHASES"].replace(0, 1)
    )
    data["CUSTOMER_TENURE_DAYS"] = (
        pd.Timestamp("2014-06-30") - data["DT_CUSTOMER"]
    ).dt.days

    y = data.pop("RESPONSE").astype(int)
    X = data.drop(
        columns=["ID", "YEAR_BIRTH", "DT_CUSTOMER", "Z_COSTCONTACT", "Z_REVENUE"]
    )
    groups = pd.util.hash_pandas_object(X, index=False)
    return X, y, groups


def make_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_columns = X.select_dtypes(include="number").columns.tolist()
    categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric, numeric_columns),
        ("categorical", categorical, categorical_columns),
    ])


def evaluate(name: str, model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    metrics = {
        "model": name,
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }
    print(pd.Series(metrics).to_string())
    return metrics


def run(df: pd.DataFrame) -> pd.DataFrame:
    X, y, groups = build_features(df)
    holdout = StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    )
    train_index, test_index = next(holdout.split(X, y, groups))
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    groups_train = groups.iloc[train_index]
    groups_test = groups.iloc[test_index]
    if set(groups_train).intersection(groups_test):
        raise RuntimeError("Um perfil de features apareceu no treino e no teste.")
    print(
        f"holdout train={len(train_index)} test={len(test_index)} "
        f"positive_test={int(y_test.sum())} ({y_test.mean():.2%})"
    )

    reference = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    reference.fit(X_train, y_train)
    results = [evaluate("reference", reference, X_test, y_test)]

    tuned = Pipeline([
        ("preprocessor", make_preprocessor(X_train)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])
    search = GridSearchCV(
        tuned,
        param_grid={
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [3, 5, 8, None],
            "model__min_samples_leaf": [1, 5, 10, 20],
            "model__class_weight": [None, "balanced"],
        },
        scoring="roc_auc",
        cv=StratifiedGroupKFold(
            n_splits=5, shuffle=True, random_state=RANDOM_STATE
        ),
        n_jobs=-1,
        refit=True,
    )
    search.fit(X_train, y_train, groups=groups_train)
    print("best_params")
    print(search.best_params_)
    results.append(evaluate("tuned", search.best_estimator_, X_test, y_test))
    return pd.DataFrame(results)




## 2. EDA

Antes de modelar, medimos o que pode afetar a decisão: quantos clientes responderam, onde faltam dados e quantos têm o mesmo perfil de preditores. A resposta é pouco frequente, então uma acurácia simples poderia esconder um modelo que quase nunca encontra quem responde. Calculamos a taxa global de RESPONSE, mas não comparamos resultados por perfil repetido antes da separação.

In [ ]:
eda = df.copy()
eda.columns = [str(column).upper() for column in eda.columns]
source_columns = len(eda.columns)
X_eda, y_eda, groups_eda = build_features(eda)
profile_sizes = pd.Series(groups_eda).value_counts()
positive = int(y_eda.sum())
summary = pd.DataFrame({
    'Indicador': ['Linhas', 'Colunas da query', 'Resposta positiva', 'INCOME nulo', 'Linhas completas duplicadas', 'IDs duplicados', 'Perfis de features repetidos', 'Linhas excedentes nesses perfis'],
    'Resultado': [
        len(eda), source_columns, f'{positive} ({positive / len(eda):.2%})',
        int(eda['INCOME'].isna().sum()), int(eda.duplicated().sum()), int(eda['ID'].duplicated().sum()),
        int((profile_sizes > 1).sum()), int((profile_sizes - 1).clip(lower=0).sum())
    ],
})
display(summary)
campaign_distribution = (
    X_eda['CAMPAIGNS_ACCEPTED'].value_counts().sort_index()
    .rename_axis('CAMPAIGNS_ACCEPTED').reset_index(name='CLIENTES')
)
display(campaign_distribution)

unique_profiles = int(profile_sizes.size)
extra_profile_rows = int((profile_sizes - 1).clip(lower=0).sum())
response_counts = y_eda.value_counts().reindex([0, 1], fill_value=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#6B7280', '#ED1165']
bars = axes[0].bar(['Não respondeu', 'Respondeu'], response_counts.values, color=colors)
axes[0].set_title('Resposta à campanha')
axes[0].set_ylabel('Clientes')
axes[0].set_ylim(0, max(response_counts.values) * 1.18)
for bar, value in zip(bars, response_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, value, f'{value:,}\n{value/len(y_eda):.1%}', ha='center', va='bottom')
bars = axes[1].bar(['Perfis distintos', 'Linhas excedentes'], [unique_profiles, extra_profile_rows], color=['#2D2D2D', '#ED1165'])
axes[1].set_title('Repetição dos preditores')
axes[1].set_ylabel('Registros')
axes[1].set_ylim(0, unique_profiles * 1.18)
for bar, value in zip(bars, [unique_profiles, extra_profile_rows]):
    axes[1].text(bar.get_x() + bar.get_width()/2, value, f'{value:,}', ha='center', va='bottom')
axes[1].text(0.5, -0.22, f'{int((profile_sizes > 1).sum())} perfis aparecem mais de uma vez', transform=axes[1].transAxes, ha='center')
fig.tight_layout()
plt.show()

## 3. Feature Engineering

AGE_AT_2014 aproxima a idade; TOTAL_CHILDREN resume a composição familiar; TOTAL_SPEND e TOTAL_PURCHASES medem valor e atividade de compra; CAMPAIGNS_ACCEPTED resume respostas a campanhas anteriores; WEB_PURCHASE_SHARE representa a preferência de canal; CUSTOMER_TENURE_DAYS mede o tempo de relacionamento. Essas variáveis resumem comportamento, valor e contexto familiar em sinais que uma equipe de produto consegue discutir. Nenhuma usa RESPONSE. Como os perfis idênticos em X poderiam inflar a avaliação, mantemos cada perfil inteiro no treino ou no teste.

Nosso alvo é prever uma resposta observada no histórico, não provar que a campanha causou a compra. Por isso, trataríamos o score como uma forma de priorizar um teste de campanha, não como uma regra automática de envio.

## 4. Comparação justa

O holdout usa o primeiro fold de StratifiedGroupKFold com cinco partes, preservando aproximadamente 80/20 e a proporção de RESPONSE sem dividir perfis repetidos. A validação cruzada também é por grupos e ocorre somente no treino. Imputação e one-hot encoding ficam dentro do Pipeline. A árvore de referência e a árvore ajustada usam o mesmo teste reservado, para que a comparação seja direta.

In [ ]:
metrics = run(df)
metrics

## 5. O que os resultados dizem

ROC AUC resume o quanto a árvore consegue ordenar clientes que responderam acima dos que não responderam. Precision e Recall ajudam a discutir a quantidade e a qualidade dos contatos selecionados; F1 resume o equilíbrio entre os dois. Não esperamos que todas as métricas melhorem ao mesmo tempo. O limiar de decisão deve considerar capacidade de contato e custo de uma resposta perdida.

Antes de usar o score em uma campanha real, definiríamos esse limiar e compararíamos um grupo de teste com um grupo de controle. Assim, separaríamos a previsão de resposta do efeito que a campanha realmente causou.

In [ ]:
metric_columns = ['precision', 'recall', 'f1', 'roc_auc']
metric_labels = {'precision': 'Precisão', 'recall': 'Recall', 'f1': 'F1', 'roc_auc': 'ROC AUC'}
metric_names = {'reference': 'Referência', 'tuned': 'Ajustada'}
plot_metrics = metrics.set_index('model')[metric_columns].rename(index=metric_names, columns=metric_labels)
ax = plot_metrics.plot(kind='bar', figsize=(10, 5), color=['#6B7280', '#ED1165', '#F59E0B', '#2D2D2D'])
ax.set_title('Árvores no mesmo teste reservado')
ax.set_ylabel('Pontuação')
ax.set_xlabel('')
ax.set_ylim(0, 1)
ax.legend(title='Métrica', ncol=2, loc='upper left')
ax.grid(axis='y', alpha=0.2)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

reference_auc = float(metrics.loc[metrics['model'] == 'reference', 'roc_auc'].iloc[0])
tuned_auc = float(metrics.loc[metrics['model'] == 'tuned', 'roc_auc'].iloc[0])
reference_precision = float(metrics.loc[metrics['model'] == 'reference', 'precision'].iloc[0])
tuned_precision = float(metrics.loc[metrics['model'] == 'tuned', 'precision'].iloc[0])
reference_recall = float(metrics.loc[metrics['model'] == 'reference', 'recall'].iloc[0])
tuned_recall = float(metrics.loc[metrics['model'] == 'tuned', 'recall'].iloc[0])
print(f'ROC AUC: {reference_auc:.4f} → {tuned_auc:.4f} ({tuned_auc-reference_auc:+.4f})')
print(f'Precisão: {reference_precision:.4f} → {tuned_precision:.4f}; Recall: {reference_recall:.4f} → {tuned_recall:.4f}')

## 6. Leitura para o negócio

Uma árvore com melhor ROC AUC pode ordenar melhor os clientes para uma lista de teste, mas não prova que uma campanha aumentará compras. A decisão seguinte é escolher quantos clientes contatar e validar o resultado com um grupo de controle.

Os resultados salvos devem vir da execução Oracle deste notebook. Não substitua essa execução pelas métricas de uma validação local.